In [ ]:
import os
import time
import logging
import random
import pandas as pd
import numpy as np
from typing import List, Optional, Tuple
from pathlib import Path
from tqdm import tqdm


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("TGTO-Core")

class TGTODefender:
    """
    Template-Guided Traffic Obfuscation (TGTO) Engine.
    Refactored for performance and maintainability.
    """

    def __init__(self, start_column: int = 103):
        """
        Args:
            start_column (int): Index where traffic features begin (e.g., 103).
        """
        self.start_col = start_column
        logger.info(f"TGTO Defender initialized. Feature start index: {self.start_col}")

    def _load_data(self, file_path: str) -> pd.DataFrame:
        try:
            return pd.read_csv(file_path, header=None)
        except Exception as e:
            logger.error(f"Failed to load file {file_path}: {e}")
            raise

    def morph_trace(self, origin_row: pd.Series, template_row: pd.Series) -> List[int]:

        
        origin_seq = origin_row[self.start_col:].tolist()
        template_seq = template_row[self.start_col:].tolist()

        modified_seq = []
        origin_ptr = 0
        origin_len = len(origin_seq)

        
        for c_val in template_seq:
           
            if origin_ptr < origin_len and origin_seq[origin_ptr] == c_val:
                modified_seq.append(origin_seq[origin_ptr])
                origin_ptr += 1
          
            elif c_val in [-1, 1]:
                modified_seq.append(c_val)
         
            else:
                continue

     
        if origin_ptr < origin_len:
            modified_seq.extend(origin_seq[origin_ptr:])


        final_seq = modified_seq[:origin_len]
        
      
        if len(final_seq) < origin_len:
            final_seq.extend([0] * (origin_len - len(final_seq)))

        return final_seq

    def apply_defense_batch(self, origin_path: str, template_path: str, output_path: str):
        logger.info(f"Processing: {Path(origin_path).name}")
        
        if not os.path.exists(origin_path) or not os.path.exists(template_path):
            logger.error("Input files not found.")
            return

      
        df_origin = self._load_data(origin_path)
        df_template = self._load_data(template_path)
        
      
        final_data = df_origin.copy()
        
        
        num_rows = len(df_origin)
        num_templates = len(df_template)
        
      
        for i in tqdm(range(num_rows), desc="Morphing Traces"):
            origin_row = df_origin.iloc[i]
            
          
            rand_idx = random.randint(0, num_templates - 1)
            template_row = df_template.iloc[rand_idx]
            
           
            morphed_features = self.morph_trace(origin_row, template_row)
            
    
            final_data.iloc[i, self.start_col:] = morphed_features
            
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        final_data.to_csv(output_path, header=False, index=False)
        logger.info(f"Saved to: {output_path}")

# ==========================================
# Main Execution
# ==========================================
if __name__ == "__main__":

    CONFIG = {
        "base_dir": "G:/Defense_code/dataset/tor-2tab-100way",
        "confusion_dir_name": "confusion",  
        "start_column": 103,
        "validation_range": [10,20,30,40]  
    }
    
    defender = TGTODefender(start_column=CONFIG["start_column"])
    
    start_time = time.time()
    
    for i in CONFIG["validation_range"]:
        origin_file = f"{CONFIG['base_dir']}/tor-2tab-finetuning100way{i}shot.csv"
        template_file = f"{CONFIG['base_dir']}/{CONFIG['confusion_dir_name']}/confusion-2.csv"
        output_file = f"{CONFIG['base_dir']}/{CONFIG['confusion_dir_name']}/tor-2tab-confusion100way{i}shotV2.csv"
        
        try:
            defender.apply_defense_batch(origin_file, template_file, output_file)
            time.sleep(1) 
        except Exception as e:
            logger.error(f"Error processing batch {i}: {e}")
            
    logger.info(f"All tasks completed in {time.time() - start_time:.2f} seconds.")